# Hyperedges and traversal

Hyperedges model complexes or reactions. Traversal helpers make
local neighborhoods available without converting to another
graph library.


In [ ]:
import annnet as an


## Undirected complexes and directed reactions


In [ ]:
H = an.AnnNet(directed=True)
H.add_vertices(['Glc', 'ATP', 'G6P', 'ADP', 'HK1', 'PFK'])
H.add_edges(['Glc', 'ATP', 'HK1'], edge_id='enzyme_complex', directed=False)
H.add_edges(
    src=['Glc', 'ATP'],
    tgt=['G6P', 'ADP'],
    edge_id='hexokinase',
    directed=True,
    weight=2.0,
)
H.add_edges('G6P', 'PFK', edge_id='activates_pfk')

H.views.edges().select(['edge_id', 'kind', 'members', 'head', 'tail'])


## Draw the hypergraph

Graphviz represents each true hyperedge through a small square
connector node. Binary edges remain ordinary arrows.


In [ ]:
from annnet.utils import plotting

plotting.plot(H, backend='graphviz', show_edge_labels=True)


## Endpoint coefficients


In [ ]:
H.set_edge_coeffs(
    'hexokinase',
    {'Glc': -1.0, 'ATP': -1.0, 'G6P': 1.0, 'ADP': 1.0},
)

col = H._edges['hexokinase'].col_idx
for vertex_id in sorted(H.vertices()):
    row = H._entities[H._resolve_entity_key(vertex_id)].row_idx
    value = H._matrix[row, col]
    if value != 0:
        print(f'{vertex_id:>4}: {value:+.1f}')


## Traverse local neighborhoods


In [ ]:
print('neighbors(Glc):', sorted(H.neighbors('Glc')))
print('successors(Glc):', sorted(H.successors('Glc')))
print('predecessors(G6P):', sorted(H.predecessors('G6P')))


## Incidence matrix


In [ ]:
import polars as pl

incidence = H.ops.vertex_incidence_matrix(values=True, sparse=True)
rows = []
for edge_id in H.edges():
    col = H._edges[edge_id].col_idx
    row = {'edge': edge_id}
    for vertex_id in sorted(H.vertices()):
        vertex_row = H._entities[H._resolve_entity_key(vertex_id)].row_idx
        row[vertex_id] = float(H._matrix[vertex_row, col])
    rows.append(row)

print('incidence shape:', incidence.shape)
print('non-zero entries:', incidence.nnz)
pl.DataFrame(rows)


The same object supports readable graph views, local traversal,
and incidence-level inspection. Use the level that matches the
question you are asking.
